# Radiomics-Based Papilledema Classification

This notebook runs the final project pipeline for binary classification of Normal vs Papilledema samples using radiomics features.

## 1. Setup

Run the cells in order. For a quick smoke test, set `TRIALS = 2`. For the final homework run, set `TRIALS = 50` or higher.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

ROOT

## 2. Load Data and Check Class Balance

In [ ]:
import pandas as pd
from radiomics_project.pipeline import load_dataset

data, feature_cols = load_dataset(ROOT / "data" / "raw")
summary = pd.DataFrame({
    "rows": [len(data)],
    "patients": [data["GroupID"].nunique()],
    "normal_rows": [(data["Target"] == 0).sum()],
    "papilledema_rows": [(data["Target"] == 1).sum()],
    "feature_count": [len(feature_cols)],
    "missing_values": [data[feature_cols].isna().sum().sum()],
})
summary

## 3. Patient-Level Split Check

The same patient must not appear in more than one split.

In [ ]:
from radiomics_project.pipeline import make_patient_level_splits

splits = make_patient_level_splits(data, seed=42)
rows = []
for split_name, idx in splits.items():
    part = data.iloc[idx]
    rows.append({
        "split": split_name,
        "rows": len(part),
        "patients": part["GroupID"].nunique(),
        "normal_patients": part.loc[part["Target"] == 0, "GroupID"].nunique(),
        "papilledema_patients": part.loc[part["Target"] == 1, "GroupID"].nunique(),
    })

split_summary = pd.DataFrame(rows)
split_summary

In [ ]:
train_groups = set(data.iloc[splits["train"]]["GroupID"])
val_groups = set(data.iloc[splits["val"]]["GroupID"])
test_groups = set(data.iloc[splits["test"]]["GroupID"])

assert train_groups.isdisjoint(val_groups)
assert train_groups.isdisjoint(test_groups)
assert val_groups.isdisjoint(test_groups)
"Patient-level split check passed"

## 4. Run the Full Pipeline

In [ ]:
from argparse import Namespace
from radiomics_project.pipeline import run

TRIALS = 50  # use 2 for a quick test, 50+ for final submission

args = Namespace(
    project_root=str(ROOT),
    trials=TRIALS,
    cv_folds=5,
    seed=42,
    correlation_threshold=0.95,
    fast=False,
    models=None,
)
run(args)

## 5. Results Tables

In [ ]:
test_results = pd.read_csv(ROOT / "outputs" / "tables" / "model_performance_test.csv")
test_results.sort_values("macro_f1", ascending=False)

In [ ]:
top_features = pd.read_csv(ROOT / "outputs" / "tables" / "top_features.csv")
top_features.head(10)

## 6. Required Figures

In [ ]:
from IPython.display import Image, display

for name in [
    "roc_curve.png",
    "precision_recall_curve.png",
    "confusion_matrix.png",
    "feature_importance.png",
    "calibration_curve.png",
    "model_comparison.png",
]:
    print(name)
    display(Image(filename=str(ROOT / "outputs" / "figures" / name)))

## 7. Discussion Prompts

- Which model achieved the highest Macro-F1?
- Did the soft voting ensemble improve over the single models?
- Did MRMR feature selection help reduce dimensionality?
- Did sigmoid calibration improve probability reliability?
- How do ROC-AUC and PR-AUC differ in this dataset?
- What are the top 10 most important radiomics features?